In [2]:
!pip install scikit-learn pandas seaborn joblib nltk

In [4]:
import pandas as pd
import numpy as np
import string
import joblib
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [10]:
import os
os.getcwd()

'C:\\Users\\91938'

In [12]:
import pandas as pd

file_path = r"C:\Users\91938\Data\final_merged_software_only_dataset.csv"

df = pd.read_csv(file_path)

print("Total rows:", len(df))
df.head()

Total rows: 9694


,Category,Resume
0,Java Developer,Professional with 1 years of experience in tec...
1,Java Developer,Professional with 4 years of experience in tec...
2,Java Developer,Professional with 3 years of experience in tec...
3,Java Developer,Professional with 1 years of experience in tec...
4,Java Developer,Professional with 1 years of experience in tec...


In [14]:
df = df.dropna(subset=['Resume'])
df['Resume'] = df['Resume'].astype(str)

print(df['Category'].value_counts())

Category
Java Developer               421
Python Developer             414
DevOps Engineer              414
React Developer              408
Cloud Engineer               406
Full Stack Developer         405
Frontend Developer           405
Data Engineer                404
Backend Developer            403
Machine Learning Engineer    402
Android Developer            402
Data Scientist               400
Automation Test Engineer     396
AWS Engineer                 392
Software Engineer            391
UI/UX Designer               389
Cybersecurity Analyst        388
MLOps Engineer               387
Node.js Developer            387
Business Analyst             354
Azure Engineer               348
Testing Engineer             347
SQL Developer                346
Golang Developer             346
Spring Boot Developer        339
Name: count, dtype: int64


In [16]:
X = df['Resume']
y = df['Category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 7755
Test size: 1939


In [18]:
logistic_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1,2),
        stop_words='english'
    )),
    ('clf', LogisticRegression(
        max_iter=2000,
        solver='lbfgs'
    ))
])

In [20]:
cv_scores_log = cross_val_score(logistic_model, X, y, cv=5)

print("Logistic 5-Fold CV Scores:", cv_scores_log)
print("Mean CV Accuracy:", cv_scores_log.mean())

Logistic 5-Fold CV Scores: [0.99535843 0.99587416 0.99226405 0.99329551 0.9122807 ]
Mean CV Accuracy: 0.9778145725324141


In [21]:
logistic_model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('clf', LogisticRegression(max_iter=2000))])

In [22]:
y_pred_log = logistic_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_log))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_log))

Test Accuracy: 0.9943269726663229

Classification Report:

                           precision    recall  f1-score   support

             AWS Engineer       1.00      1.00      1.00        78
        Android Developer       1.00      1.00      1.00        81
 Automation Test Engineer       1.00      1.00      1.00        79
           Azure Engineer       1.00      1.00      1.00        70
        Backend Developer       1.00      1.00      1.00        81
         Business Analyst       1.00      1.00      1.00        71
           Cloud Engineer       1.00      1.00      1.00        81
    Cybersecurity Analyst       1.00      1.00      1.00        78
            Data Engineer       1.00      1.00      1.00        81
           Data Scientist       1.00      1.00      1.00        80
          DevOps Engineer       1.00      1.00      1.00        83
       Frontend Developer       1.00      0.96      0.98        81
     Full Stack Developer       0.94      0.96      0.95        81
  

In [26]:
svm_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1,2),
        stop_words='english'
    )),
    ('clf', LinearSVC())
])

In [28]:
cv_scores_svm = cross_val_score(svm_model, X, y, cv=5)

print("SVM 5-Fold CV Scores:", cv_scores_svm)
print("Mean CV Accuracy:", cv_scores_svm.mean())

SVM 5-Fold CV Scores: [0.99535843 0.99638989 0.99277978 0.9948427  0.90866873]
Mean CV Accuracy: 0.9776079080691747


In [30]:
svm_model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('clf', LinearSVC())])

In [32]:
y_pred_svm = svm_model.predict(X_test)

print("Test Accuracy (SVM):", accuracy_score(y_test, y_pred_svm))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm))

Test Accuracy (SVM): 0.9958741619391439

Classification Report:

                           precision    recall  f1-score   support

             AWS Engineer       1.00      1.00      1.00        78
        Android Developer       1.00      1.00      1.00        81
 Automation Test Engineer       1.00      1.00      1.00        79
           Azure Engineer       1.00      1.00      1.00        70
        Backend Developer       1.00      1.00      1.00        81
         Business Analyst       1.00      1.00      1.00        71
           Cloud Engineer       1.00      1.00      1.00        81
    Cybersecurity Analyst       1.00      1.00      1.00        78
            Data Engineer       1.00      1.00      1.00        81
           Data Scientist       1.00      1.00      1.00        80
          DevOps Engineer       1.00      1.00      1.00        83
       Frontend Developer       1.00      0.98      0.99        81
     Full Stack Developer       0.95      0.98      0.96       

In [34]:
print("Logistic CV Mean:", cv_scores_log.mean())
print("SVM CV Mean:", cv_scores_svm.mean())

Logistic CV Mean: 0.9778145725324141
SVM CV Mean: 0.9776079080691747


In [36]:
svm_model.fit(X, y)
joblib.dump(svm_model, "final_resume_classifier.pkl")
print("Model Saved Successfully")

Model Saved Successfully


In [38]:
logistic_model.fit(X, y)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('clf', LogisticRegression(max_iter=2000))])

In [40]:
import joblib

joblib.dump(logistic_model, "final_resume_classifier_logistic.pkl")

print("Final Logistic Model Saved Successfully")

Final Logistic Model Saved Successfully
